In [1]:
import numpy as np
import pandas as pd
import os
import time
import pprint
import itertools
import ast

from tqdm.auto import tqdm

from sklearn.base import is_regressor, clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, log_loss
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC, SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from joblib import Parallel, delayed

from pyagrum.skbn import BNClassifier
from pyagrum.lib.discreteTypeProcessor import DiscreteTypeProcessor

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_parquet(os.path.join("..", "out", "processed", "train.parquet"))
val = pd.read_parquet(os.path.join("..", "out", "processed", "val.parquet"))
test = pd.read_parquet(os.path.join("..", "out", "processed", "test.parquet"))

meta_val = val[["open_time"]].copy()
meta_test = test[["open_time"]].copy()

train = train.drop(columns=["open_time"])
val = val.drop(columns=["open_time"])
test = test.drop(columns=["open_time"])

X_train = train.drop(columns=["label"])
y_train = train["label"]

X_val = val.drop(columns=["label"])
y_val = val["label"]

X_test = test.drop(columns=["label"])
y_test = test["label"]

rng = np.random.RandomState(0)
train_idx = rng.choice(len(X_train), 200_000, replace=False)
valid_idx = rng.choice(len(X_val), 30_000, replace=False)

X_train_sample = X_train.iloc[train_idx].to_numpy()
y_train_sample = y_train.iloc[train_idx].to_numpy()
X_val_sample = X_val.iloc[valid_idx].to_numpy()
y_val_sample = y_val.iloc[valid_idx].to_numpy()

# kernel SVM scales ~quadratically with rows, so it gets a 20k slice of the 200k sample
X_train_small = X_train_sample[:20_000]
y_train_small = y_train_sample[:20_000]

RESULTS = os.path.join("..", "out", "results")
os.makedirs(RESULTS, exist_ok=True)

def save(results, name):
    results.to_csv(os.path.join(RESULTS, f"{name}.csv"), index=False)

SEARCH = False  # True re-runs every hyperparameter search (~6.5 h on 14 cores); False reloads the saved out/results/*.csv

def parse(v):
    if v == "":
        return None  # None is written to CSV as an empty cell
    try:
        return ast.literal_eval(v)  # "(64, 32)" -> tuple, "0.2" -> float, "True" -> bool
    except (ValueError, SyntaxError):
        return v  # plain strings such as "sqrt"

def search(name, run):
    """Run a hyperparameter search and save it, or reload the saved one when SEARCH is False."""
    path = os.path.join(RESULTS, f"{name}.csv")
    if not SEARCH and os.path.exists(path):
        return pd.read_csv(path, keep_default_na=False, dtype=str).apply(lambda col: col.map(parse))
    results = run()
    save(results, name)
    return results

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

print(X_train.describe().loc[["mean", "std"]].T)

Training set shape: (1127670, 36) (1127670,)
Validation set shape: (188886, 36) (188886,)
Test set shape: (298959, 36) (298959,)


                               mean       std
symbol=BTCUSDT         8.404675e-02  0.277458
symbol=ETHUSDT         9.571062e-02  0.294194
symbol=BNBUSDT         9.720840e-02  0.296241
symbol=XRPUSDT         1.029530e-01  0.303898
symbol=ADAUSDT         1.097883e-01  0.312626
symbol=DOGEUSDT        9.485488e-02  0.293015
symbol=SOLUSDT         1.016264e-01  0.302156
symbol=LTCUSDT         1.044862e-01  0.305890
symbol=LINKUSDT        1.138782e-01  0.317663
symbol=AVAXUSDT        9.544725e-02  0.293832
returns                4.839153e-18  1.000000
volatility            -2.580882e-17  1.000000
upper_wick            -6.452205e-17  1.000000
lower_wick             0.000000e+00  1.000000
body_ratio             4.375401e-17  1.000000
taker_buy_ratio       -2.790579e-16  1.000000
taker_buy_ratio_16    -1.451746e-15  1.000000
relative_volume       -1.729997e-16  1.000000
normalized_volatility -1.169462e-17  1.000000
rel_trade_size        -4.597196e-17  1.000000
ret_4                  6.653836e-1

In [3]:
model_results = []
fits = {}  # name -> (fitted model, X_train it was fitted on, y_train), reused in the variability analysis
val_scores = {}  # name -> P(up) on the full validation set

def in_batches(fn, X, size=20_000):
    # KNN with k in the thousands builds a (rows x k) neighbor matrix: all 189k val rows at once ran out of memory
    return np.concatenate([fn(X[i:i + size]) for i in range(0, len(X), size)])

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return in_batches(lambda x: model.predict_proba(x)[:, 1], X)
    elif hasattr(model, "decision_function"):
        return in_batches(model.decision_function, X)
    return in_batches(model.predict, X)

def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw = in_batches(model.predict, X_val)
    predict_time = time.perf_counter() - t0

    reg = is_regressor(model)
    y_pred = (raw > 0.5).astype(int) if reg else raw
    scores = raw if reg else get_scores(model, X_val)
    train_scores = in_batches(model.predict, X_train) if reg else get_scores(model, X_train)
    # a regressor's output can fall outside [0, 1]; clip it so it can be scored as a probability
    clip = lambda p: np.clip(p, 1e-15, 1 - 1e-15)

    row = {
        "model": name,
        "fit_time": fit_time,
        "predict_time": predict_time,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, scores),
        "confusion_matrix": confusion_matrix(y_val, y_pred).tolist(),
        "train_log_loss": log_loss(y_train, clip(train_scores)),
        "log_loss": log_loss(y_val, clip(scores)),
    }
    model_results.append(row)
    fits[name] = (model, X_train, y_train)
    val_scores[name] = clip(scores)
    pprint.pprint(row)

In [4]:
# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def run_parallel(fn, combos, desc, n_jobs=10):
    """Run fn(*combo) for every combo in parallel with a progress bar that vanishes at the end.
    Needs joblib >= 1.3 for return_as="generator"."""
    gen = Parallel(n_jobs=n_jobs, return_as="generator_unordered")(
        delayed(fn)(*c) for c in combos
    )
    return list(tqdm(gen, total=len(combos), desc=desc, leave=False))

def run_serial(fn, combos, desc):
    return [fn(*c) for c in tqdm(combos, desc=desc, leave=False)]

def to_df(rows, by="val_log_loss", ascending=True):
    return pd.DataFrame(rows).sort_values(by=by, ascending=ascending).reset_index(drop=True)

def metrics(y_true, p):
    """p = probability of the positive class."""
    return {
        "log_loss": log_loss(y_true, p),
        "auc": roc_auc_score(y_true, p),
    }


# ----------------------------------------------------------------------------
# Logistic Regression
# ----------------------------------------------------------------------------
def evaluate_logistic_regression(solver, penalty, C, max_iter=1000):
    m = LogisticRegression(
        solver=solver,
        penalty=penalty,
        C=C,
        max_iter=max_iter,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_logistic_regression(solvers, penalties, Cs):
    combos = list(itertools.product(solvers, penalties, Cs))
    return to_df(run_parallel(evaluate_logistic_regression, combos, "LogReg"))


# ----------------------------------------------------------------------------
# KNN
# ----------------------------------------------------------------------------
def evaluate_knn(n_neighbors, weights, metric):
    m = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric, n_jobs=-1)
    m.fit(X_train_sample, y_train_sample)
    # train score on a 30k slice: scoring all 200k training points against themselves is slow and,
    # with weights="distance", meaningless (each point is its own neighbor at distance 0)
    p_train = m.predict_proba(X_train_sample[:30_000])[:, 1]
    p_val = m.predict_proba(X_val_sample)[:, 1]
    return {
        "n_neighbors": n_neighbors,
        "weights": weights,
        "metric": metric,
        "train_log_loss": log_loss(y_train_sample[:30_000], p_train),
        "val_log_loss": log_loss(y_val_sample, p_val),
        "train_auc": roc_auc_score(y_train_sample[:30_000], p_train),
        "val_auc": roc_auc_score(y_val_sample, p_val),
    }


def select_knn(n_neighbors, weights, metrics_):
    combos = list(itertools.product(n_neighbors, weights, metrics_))
    return to_df(run_serial(evaluate_knn, combos, "KNN"))


# ----------------------------------------------------------------------------
# Decision Tree
# ----------------------------------------------------------------------------
def evaluate_tree(criterion, max_depth, min_samples_leaf, splitter="best",
                  min_samples_split=2, max_leaf_nodes=None, min_impurity_decrease=0.0):
    m = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        splitter=splitter,
        min_samples_split=min_samples_split,
        max_leaf_nodes=max_leaf_nodes,
        min_impurity_decrease=min_impurity_decrease,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "criterion": criterion,
        "splitter": splitter,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "min_samples_split": min_samples_split,
        "max_leaf_nodes": max_leaf_nodes,
        "min_impurity_decrease": min_impurity_decrease,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_tree(criterions, max_depths, min_samples_leaves,
                splitters=("best", "random"), min_samples_split=(2, 5, 10),
                max_leaf_nodes=(None,), min_impurity_decrease=(0.0,)):
    combos = list(itertools.product(
        criterions, max_depths, min_samples_leaves,
        splitters, min_samples_split, max_leaf_nodes, min_impurity_decrease,
    ))
    return to_df(run_parallel(evaluate_tree, combos, "Tree"))


# ----------------------------------------------------------------------------
# Gaussian Naive Bayes
# ----------------------------------------------------------------------------
def evaluate_gaussian_nb(var_smoothing):
    m = GaussianNB(var_smoothing=var_smoothing)
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "var_smoothing": var_smoothing,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_gaussian_nb(var_smoothing):
    combos = [(v,) for v in var_smoothing]
    return to_df(run_parallel(evaluate_gaussian_nb, combos, "GaussianNB"))


# ----------------------------------------------------------------------------
# Random Forest
# ----------------------------------------------------------------------------
def evaluate_random_forest(n_estimators, max_features, max_depth=None):
    m = RandomForestClassifier(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "n_estimators": n_estimators,
        "max_features": max_features,
        "max_depth": max_depth,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_random_forest(n_estimators, max_features, max_depths=(None,)):
    combos = list(itertools.product(n_estimators, max_features, max_depths))
    rows = run_parallel(evaluate_random_forest, combos, "RandomForest", n_jobs=4)
    return to_df(rows)


# ----------------------------------------------------------------------------
# AdaBoost
# ----------------------------------------------------------------------------
def adaboost_evaluate(estimator, n_estimators, learning_rate):
    m = AdaBoostClassifier(estimator=estimator, n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "base_depth": estimator.max_depth,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }




def adaboost_select(estimators, n_estimators, learning_rates):
    combos = list(itertools.product(estimators, n_estimators, learning_rates))
    return to_df(run_parallel(adaboost_evaluate, combos, "AdaBoost"))


# ----------------------------------------------------------------------------
# Linear SVM
# ----------------------------------------------------------------------------

def evaluate_svm(C, penalty, loss, dual, tol, class_weight=None, max_iter=1000, random_state=42):
    m = LinearSVC(C=C, penalty=penalty, loss=loss, dual=dual,
                  max_iter=max_iter, random_state=random_state, tol=tol, class_weight=class_weight)
    m = CalibratedClassifierCV(m, method="sigmoid", cv=3)
    m.fit(X_train, y_train)
    s_train = m.predict_proba(X_train)[:, 1]
    s_val = m.predict_proba(X_val)[:, 1]
    return {
        "C": C,
        "penalty": penalty,
        "loss": loss,
        "dual": dual,
        "train_log_loss": log_loss(y_train, s_train),
        "val_log_loss": log_loss(y_val, s_val),
        "train_auc": roc_auc_score(y_train, s_train),
        "tol": tol,
        "class_weight": class_weight,
        "val_auc": roc_auc_score(y_val, s_val),
    }


def select_svm(Cs, penalties, losses, duals, tols):
    combos = list(itertools.product(Cs, penalties, losses, duals, tols))
    return to_df(run_parallel(evaluate_svm, combos, "SVM"))


# ----------------------------------------------------------------------------
# MLP
# ----------------------------------------------------------------------------
def evaluate_mlp(hidden_layer_sizes, activation, solver, alpha, learning_rate, learning_rate_init, max_iter=200):
    # trained on the 200k sample with early stopping: the full 1.1M rows x 10 workers ran out of memory
    m = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        alpha=alpha,
        learning_rate=learning_rate,
        learning_rate_init=learning_rate_init,
        max_iter=max_iter,
        early_stopping=True,
        random_state=42,
    )
    m.fit(X_train_sample, y_train_sample)
    p_train = m.predict_proba(X_train_sample)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "hidden_layer_sizes": hidden_layer_sizes,
        "activation": activation,
        "solver": solver,
        "alpha": alpha,
        "learning_rate_init": learning_rate_init,
        "learning_rate": learning_rate,
        "n_iter": m.n_iter_,
        "train_log_loss": log_loss(y_train_sample, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train_sample, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_mlp(hidden_layer_sizes_list, activations, solvers, alphas, learning_rates, learning_rate_inits=(0.001,)):
    combos = list(itertools.product(hidden_layer_sizes_list, activations, solvers, alphas, learning_rates, learning_rate_inits))
    return to_df(run_parallel(evaluate_mlp, combos, "MLP", n_jobs=6))


def evaluate_mlp_full(hidden_layer_sizes, alpha):
    # second round on all 1.1M rows: the 200k sample was only needed to run 48 fits in parallel
    m = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        solver="sgd",
        learning_rate="constant",
        learning_rate_init=0.001,
        alpha=alpha,
        max_iter=200,
        early_stopping=True,
        random_state=42,
    )
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "hidden_layer_sizes": hidden_layer_sizes,
        "alpha": alpha,
        "n_iter": m.n_iter_,
        "train_log_loss": log_loss(y_train, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_mlp_full(hidden_layer_sizes_list, alphas):
    combos = list(itertools.product(hidden_layer_sizes_list, alphas))
    return to_df(run_parallel(evaluate_mlp_full, combos, "MLP full data", n_jobs=9))


# ----------------------------------------------------------------------------
# Kernel (RBF) SVM
# ----------------------------------------------------------------------------
def evaluate_kernel_svm(C, gamma):
    m = SVC(kernel="rbf", C=C, gamma=gamma, probability=True, random_state=42)
    m.fit(X_train_small, y_train_small)
    p_train = m.predict_proba(X_train_small)[:, 1]
    p_val = m.predict_proba(X_val_sample)[:, 1]
    return {
        "C": C,
        "gamma": gamma,
        "n_support": int(m.n_support_.sum()),
        "train_log_loss": log_loss(y_train_small, p_train),
        "val_log_loss": log_loss(y_val_sample, p_val),
        "train_auc": roc_auc_score(y_train_small, p_train),
        "val_auc": roc_auc_score(y_val_sample, p_val),
    }


def select_kernel_svm(Cs, gammas):
    combos = list(itertools.product(Cs, gammas))
    return to_df(run_parallel(evaluate_kernel_svm, combos, "KernelSVM"))


# ----------------------------------------------------------------------------
# Bayesian Network
# ----------------------------------------------------------------------------
def round6(X):
    # Rounding: dow_cos holds 4 real values stored as 7 floats (cos(Mon) == cos(Sat) up to 1e-16); pyagrum writes
    # values out as text, the pairs collide, and fit crashes with "Duplicate element".
    # Plain array: pyagrum fitted on an array misreads a DataFrame at predict time ("Bad label" with a whole row).
    return np.round(np.asarray(X, dtype=float), 6)


def make_bn(learning_method, discretization, n_bins):
    return make_pipeline(
        FunctionTransformer(round6),
        BNClassifier(
            type_processor=DiscreteTypeProcessor(defaultDiscretizationMethod=discretization, defaultNumberOfBins=n_bins),
            learningMethod=learning_method,
        ),
    )


def evaluate_bn(learning_method, discretization, n_bins):
    # round one: 200k sample, so that five fits can run in parallel (a full-data fit needs ~9 GB of memory)
    m = make_bn(learning_method, discretization, n_bins)
    m.fit(X_train_sample, y_train_sample)
    p_train = m.predict_proba(X_train_sample)[:, 1]
    p_val = m.predict_proba(X_val_sample)[:, 1]
    return {
        "learning_method": learning_method,
        "discretization": discretization,
        "n_bins": n_bins,
        "train_log_loss": log_loss(y_train_sample, p_train),
        "val_log_loss": log_loss(y_val_sample, p_val),
        "train_auc": roc_auc_score(y_train_sample, p_train),
        "val_auc": roc_auc_score(y_val_sample, p_val),
    }


def select_bn(learning_methods, discretizations, n_bins):
    combos = list(itertools.product(learning_methods, discretizations, n_bins))
    return to_df(run_parallel(evaluate_bn, combos, "BayesNet", n_jobs=5))


def evaluate_bn_full(learning_method, discretization, n_bins):
    # round two: all 1.1M rows. The training score uses the 200k sample rows (all training rows), which saves
    # predicting 1.1M rows per setting
    m = make_bn(learning_method, discretization, n_bins)
    m.fit(X_train, y_train)
    p_train = m.predict_proba(X_train_sample)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    return {
        "learning_method": learning_method,
        "discretization": discretization,
        "n_bins": n_bins,
        "train_log_loss": log_loss(y_train_sample, p_train),
        "val_log_loss": log_loss(y_val, p_val),
        "train_auc": roc_auc_score(y_train_sample, p_train),
        "val_auc": roc_auc_score(y_val, p_val),
    }


def select_bn_full(learning_methods, discretizations, n_bins):
    # one fit at a time: each full-data fit peaks at ~9 GB of memory
    combos = list(itertools.product(learning_methods, discretizations, n_bins))
    return to_df(run_serial(evaluate_bn_full, combos, "BayesNet full data"))

# Majority Classifier

In [5]:
# "prior" predicts the training share of up moves (~50.7%) for every row. "most_frequent" would predict up
# with 100% confidence, and log loss punishes every wrong 100%-confident answer with ~36, giving ~17.8.
evaluate_model('Majority', DummyClassifier(strategy='prior'), X_train, y_train, X_val, y_val)

{'accuracy': 0.5062100949779232,
 'confusion_matrix': [[0, 93270], [0, 95616]],
 'f1': 0.6721639918172807,
 'fit_time': 0.02280062501085922,
 'log_loss': 0.6930710274306106,
 'model': 'Majority',
 'precision': 0.5062100949779232,
 'predict_time': 0.0005709580145776272,
 'recall': 1.0,
 'roc_auc': 0.5,
 'train_log_loss': 0.6930516862149106}


# Linear Regression

In [6]:
evaluate_model('Linear Regression', LinearRegression(), X_train, y_train, X_val, y_val)

{'accuracy': 0.5270533549336637,
 'confusion_matrix': [[43584, 49686], [39647, 55969]],
 'f1': 0.556155630965216,
 'fit_time': 0.313166166015435,
 'log_loss': 0.6908403255373237,
 'model': 'Linear Regression',
 'precision': 0.529733566797596,
 'predict_time': 0.00804483302636072,
 'recall': 0.5853518239625167,
 'roc_auc': 0.5378091515753877,
 'train_log_loss': 0.6895841546359907}


# Logistic Regression

In [7]:
Cs = [0.00001, 0.0001, 0.001, 0.01]
results = search("logistic_regression", lambda: pd.concat([
    select_logistic_regression(solvers=['lbfgs', 'newton-cg', 'sag'], penalties=['l2', None], Cs=Cs),
    select_logistic_regression(solvers=['liblinear'], penalties=['l1', 'l2'], Cs=Cs),
    select_logistic_regression(solvers=['saga'], penalties=['l1', 'l2', 'elasticnet', None], Cs=Cs),
]).sort_values(by="val_log_loss").reset_index(drop=True))
print(results.head(10).to_string())

      solver     penalty       C  train_log_loss  val_log_loss  train_auc   val_auc
0      lbfgs          l2  0.0001        0.689594      0.690824   0.548465  0.537722
1  liblinear          l1  0.0100        0.689579      0.690825   0.548443  0.537821
2  liblinear          l2  0.0001        0.689593      0.690828   0.548458  0.537697
3       saga          l1  0.0100        0.689579      0.690828   0.548444  0.537802
4  newton-cg          l2  0.0001        0.689593      0.690828   0.548459  0.537700
5       saga  elasticnet  0.0001        0.689593      0.690829   0.548458  0.537699
6       saga          l2  0.0001        0.689593      0.690829   0.548458  0.537699
7        sag          l2  0.0001        0.689593      0.690829   0.548458  0.537699
8  newton-cg          l2  0.0010        0.689571      0.690831   0.548451  0.537846
9  liblinear          l2  0.0010        0.689571      0.690832   0.548452  0.537845


In [8]:
best = results.iloc[0]
evaluate_model("Logistic Regression", LogisticRegression(solver=best.solver, penalty=best.penalty, C=best.C, max_iter=1000, random_state=42), X_train, y_train, X_val, y_val)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


{'accuracy': 0.5273604184534587,
 'confusion_matrix': [[43264, 50006], [39269, 56347]],
 'f1': 0.5579767191994811,
 'fit_time': 0.33140129200182855,
 'log_loss': 0.6908243079297078,
 'model': 'Logistic Regression',
 'precision': 0.5298111007681965,
 'predict_time': 0.008365375048015267,
 'recall': 0.5893051372155288,
 'roc_auc': 0.5377215134437898,
 'train_log_loss': 0.6895935246070064}


# KNN

In [9]:
# One neighbor search with the largest k, then every smaller k is a prefix of the same sorted neighbor list
ks = [100, 250, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 5000]

def knn_k_sweep():
    nn = NearestNeighbors(n_neighbors=max(ks), n_jobs=-1).fit(X_train_sample)
    distances, indices = nn.kneighbors(X_val_sample)
    neighbor_labels = y_train_sample[indices]

    rows = []
    for k in ks:
        scores = neighbor_labels[:, :k].mean(axis=1)
        rows.append({
            'k': k,
            'val_log_loss': log_loss(y_val_sample, np.clip(scores, 1e-15, 1 - 1e-15)),
            'val_auc': roc_auc_score(y_val_sample, scores),
        })
    return to_df(rows)

knn_k = search("knn_k", knn_k_sweep)
print(knn_k.to_string())

       k  val_log_loss   val_auc
0   3500      0.690167  0.542494
1   2500      0.690169  0.541865
2   2000      0.690197  0.541370
3   3000      0.690198  0.541988
4   1500      0.690211  0.541224
5   4000      0.690222  0.542226
6   5000      0.690297  0.541640
7   1000      0.690306  0.540457
8    500      0.690829  0.538400
9    250      0.691957  0.532986
10   100      0.695107  0.527060


In [10]:
results = search("knn", lambda: select_knn(
    n_neighbors=[int(knn_k.iloc[0].k)],
    weights=['uniform', 'distance'],
    metrics_=['euclidean', 'manhattan'],
))
print(results.to_string())

   n_neighbors   weights     metric  train_log_loss  val_log_loss  train_auc   val_auc
0         3500  distance  euclidean    9.918999e-06      0.690152   1.000000  0.542494
1         3500   uniform  euclidean    6.892179e-01      0.690167   0.552299  0.542494
2         3500  distance  manhattan    2.220446e-16      0.690363   1.000000  0.540031
3         3500   uniform  manhattan    6.886729e-01      0.690380   0.557190  0.539968


In [11]:
best = results.iloc[0]
evaluate_model('KNN', KNeighborsClassifier(n_neighbors=int(best.n_neighbors), weights=best.weights, metric=best.metric, n_jobs=-1), X_train_sample, y_train_sample, X_val, y_val)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


... (11 outputs left out)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


{'accuracy': 0.5262539309424732,
 'confusion_matrix': [[45197, 48073], [41411, 54205]],
 'f1': 0.5478185291115446,
 'fit_time': 0.011335249990224838,
 'log_loss': 0.6907435937619691,
 'model': 'KNN',
 'precision': 0.5299771211795303,
 'predict_time': 55.821441667038016,
 'recall': 0.5669030287817939,
 'roc_auc': 0.5376816743718019,
 'train_log_loss': 1.007273590568599e-05}


# Decision Tree

In [12]:
# entropy and log_loss are the same criterion in sklearn; min_samples_split only matters below 2 x min_samples_leaf
results = search("decision_tree", lambda: select_tree(
   criterions=['entropy', 'gini'],
   max_depths=[2, 3, 4, 5, 6],
   min_samples_leaves=[500, 1000, 2000, 3000],
   splitters=['best', 'random'],
   min_samples_split=[2],
   max_leaf_nodes=[None],
   min_impurity_decrease=[0.0],
   ))
print(results.head(10).to_string())

  criterion splitter  max_depth  min_samples_leaf  min_samples_split max_leaf_nodes  min_impurity_decrease  train_log_loss  val_log_loss  train_auc   val_auc
0   entropy     best          4              2000                  2           None                    0.0        0.688985      0.691937   0.550443  0.531596
1   entropy     best          4              3000                  2           None                    0.0        0.688985      0.691937   0.550443  0.531596
2      gini     best          4              2000                  2           None                    0.0        0.688985      0.691937   0.550443  0.531596
3      gini     best          4              3000                  2           None                    0.0        0.688985      0.691937   0.550443  0.531596
4      gini     best          4              1000                  2           None                    0.0        0.688983      0.691948   0.550442  0.531596
5   entropy     best          4              1000   

In [13]:
best = results.iloc[0]
evaluate_model('Decision Tree', DecisionTreeClassifier(criterion=best.criterion, splitter=best.splitter, max_depth=int(best.max_depth), min_samples_leaf=int(best.min_samples_leaf), random_state=42), X_train, y_train, X_val, y_val)

{'accuracy': 0.522902703217814,
 'confusion_matrix': [[51043, 42227], [47890, 47726]],
 'f1': 0.5143747069823085,
 'fit_time': 8.656709083996248,
 'log_loss': 0.6919371257296705,
 'model': 'Decision Tree',
 'precision': 0.5305659622247173,
 'predict_time': 0.013281874998938292,
 'recall': 0.4991424029451138,
 'roc_auc': 0.5315958407066492,
 'train_log_loss': 0.6889845324850463}


# Random Forest

In [14]:
# the first search peaked at the deepest value tried (8), so deeper trees are added; 400 trees never beat 100-200
results = search("random_forest", lambda: select_random_forest(
    n_estimators=[100, 200],
    max_features=['sqrt', 0.2],
    max_depths=[8, 10, 12, 16],
))
print(results.head(10).to_string())

   n_estimators max_features  max_depth  train_log_loss  val_log_loss  train_auc   val_auc
0           100         sqrt          8        0.684907      0.690679   0.576768  0.538751
1           200          0.2          8        0.684534      0.690695   0.578509  0.538842
2           200         sqrt          8        0.684858      0.690701   0.577241  0.538590
3           100          0.2          8        0.684520      0.690706   0.578593  0.538757
4           200          0.2         10        0.678838      0.690711   0.609751  0.538975
5           200         sqrt         10        0.679481      0.690715   0.606847  0.538921
6           100          0.2         10        0.678792      0.690717   0.609781  0.538989
7           100         sqrt         10        0.679562      0.690749   0.606173  0.538777
8           200         sqrt         12        0.668701      0.690777   0.662875  0.538823
9           200          0.2         12        0.667416      0.690788   0.668070  0.538988

In [15]:
best = results.iloc[0]
evaluate_model('Random Forest', RandomForestClassifier(n_estimators=int(best.n_estimators), max_features=best.max_features, max_depth=int(best.max_depth), random_state=42, n_jobs=-1), X_train, y_train, X_val, y_val)

{'accuracy': 0.5277786601442139,
 'confusion_matrix': [[47194, 46076], [43120, 52496]],
 'f1': 0.5406719261746349,
 'fit_time': 19.25179020903306,
 'log_loss': 0.6906793500225281,
 'model': 'Random Forest',
 'precision': 0.5325650286085298,
 'predict_time': 0.17311483400408179,
 'recall': 0.5490294511378849,
 'roc_auc': 0.5387512364286853,
 'train_log_loss': 0.6849068862020893}


# Naive Bayes

In [16]:
results = search("naive_bayes", lambda: select_gaussian_nb(
    var_smoothing=[1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 20, 50, 100, 200]
))
print(results.head(10).to_string())

   var_smoothing  train_log_loss  val_log_loss  train_auc   val_auc
0        10.0000        0.691586      0.692103   0.546570  0.535133
1        20.0000        0.692130      0.692422   0.546591  0.534966
2        50.0000        0.692634      0.692766   0.546564  0.534894
3       100.0000        0.692834      0.692910   0.546550  0.534880
4       200.0000        0.692941      0.692989   0.546542  0.534876
5         1.0000        0.712818      0.705514   0.544433  0.534740
6         0.1000        0.802785      0.757545   0.541794  0.532754
7         0.0100        0.829269      0.773140   0.541303  0.532313
8         0.0010        0.832453      0.775202   0.541113  0.532172
9         0.0001        0.832778      0.775416   0.541088  0.532153


In [17]:
best = results.iloc[0]
evaluate_model("Naive Bayes", GaussianNB(var_smoothing=best.var_smoothing), X_train, y_train, X_val, y_val)

{'accuracy': 0.5162743665491355,
 'confusion_matrix': [[72061, 21209], [70160, 25456]],
 'f1': 0.35782711676190077,
 'fit_time': 0.19520249997731298,
 'log_loss': 0.6921025471995647,
 'model': 'Naive Bayes',
 'precision': 0.5455051966141647,
 'predict_time': 0.026861333986744285,
 'recall': 0.26623159303882193,
 'roc_auc': 0.5351330642429624,
 'train_log_loss': 0.6915863270545494}


# AdaBoost

In [18]:
results = search("adaboost", lambda: adaboost_select(
    estimators=[DecisionTreeClassifier(max_depth=1, random_state=42), DecisionTreeClassifier(max_depth=2, random_state=42)],
    n_estimators=[400, 800, 1000, 1500],
    learning_rates=[0.1, 0.3, 0.5, 1.0]
))
print(results.head(10).to_string())

   base_depth  n_estimators  learning_rate  train_log_loss  val_log_loss  train_auc   val_auc
0           1          1500            1.0        0.688316      0.691024   0.559619  0.536212
1           1          1000            1.0        0.688006      0.691064   0.558929  0.536340
2           2          1000            1.0        0.687471      0.691080   0.570053  0.535423
3           2           800            1.0        0.687190      0.691081   0.569354  0.535181
4           1           800            1.0        0.687882      0.691169   0.558384  0.536834
5           2          1500            1.0        0.687926      0.691170   0.570536  0.535252
6           2          1500            0.5        0.686955      0.691320   0.562603  0.537117
7           2           400            1.0        0.686596      0.691558   0.565198  0.534557
8           2          1000            0.5        0.687297      0.692264   0.560848  0.537291
9           1           400            1.0        0.688157  

In [19]:
best = results.iloc[0]
evaluate_model('AdaBoost', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=int(best.base_depth), random_state=42), n_estimators=int(best.n_estimators), learning_rate=best.learning_rate, random_state=42), X_train, y_train, X_val, y_val)

{'accuracy': 0.525359211376174,
 'confusion_matrix': [[48022, 45248], [44405, 51211]],
 'f1': 0.5332396199401276,
 'fit_time': 3198.025229333027,
 'log_loss': 0.6910238302216933,
 'model': 'AdaBoost',
 'precision': 0.53090950559305,
 'predict_time': 7.150809874990955,
 'recall': 0.5355902777777778,
 'roc_auc': 0.5362123302679644,
 'train_log_loss': 0.6883164928384036}


# SVM

In [20]:
results = search("linear_svm", lambda: select_svm(
    Cs=[0.01, 0.1, 1.0, 10.0, 100.0],
    penalties=['l2'],
    losses=['squared_hinge'],
    duals=[True],
    tols=[1e-4, 1e-3, 1e-2, 1e-1]
))
print(results.head(10).to_string())

      C penalty           loss  dual  train_log_loss  val_log_loss  train_auc     tol class_weight   val_auc
0  1.00      l2  squared_hinge  True        0.689922      0.690885   0.548312  0.1000         None  0.537876
1  1.00      l2  squared_hinge  True        0.689922      0.690885   0.548312  0.0001         None  0.537876
2  1.00      l2  squared_hinge  True        0.689922      0.690885   0.548312  0.0100         None  0.537876
3  1.00      l2  squared_hinge  True        0.689922      0.690885   0.548312  0.0010         None  0.537876
4  0.10      l2  squared_hinge  True        0.689924      0.690890   0.548283  0.1000         None  0.537785
5  0.01      l2  squared_hinge  True        0.689924      0.690890   0.548282  0.1000         None  0.537781
6  0.01      l2  squared_hinge  True        0.689924      0.690890   0.548280  0.0010         None  0.537779
7  0.01      l2  squared_hinge  True        0.689924      0.690890   0.548280  0.0001         None  0.537779
8  0.01      l2  sq

In [21]:
best = results.iloc[0]
evaluate_model('Linear SVM', CalibratedClassifierCV(LinearSVC(C=best.C, penalty=best.penalty, loss=best.loss, dual=best.dual, tol=best.tol, random_state=42), method="sigmoid", cv=3), X_train, y_train, X_val, y_val)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


{'accuracy': 0.527948074499963,
 'confusion_matrix': [[43111, 50159], [39005, 56611]],
 'f1': 0.5594359293627029,
 'fit_time': 445.1137424589833,
 'log_loss': 0.6908849553124035,
 'model': 'Linear SVM',
 'precision': 0.5302144797227686,
 'predict_time': 0.034212667029350996,
 'recall': 0.5920661813922357,
 'roc_auc': 0.5378756200734822,
 'train_log_loss': 0.6899216484432312}


# Neural Network

In [22]:
# Round one: 200k-row sample (evaluate_mlp fits on X_train_sample), 48 settings in parallel.
# With SEARCH = False this reloads out/results/mlp.csv instead of retraining.
results = search("mlp", lambda: pd.concat([
    select_mlp(
        hidden_layer_sizes_list=[(64,), (64, 32), (128, 64)],
        activations=["relu"],
        solvers=["sgd"],
        alphas=[1e-4, 1e-2],
        learning_rates=["constant", "invscaling", "adaptive"],
        learning_rate_inits=(0.001, 0.01)
    ),
    select_mlp(
        hidden_layer_sizes_list=[(64,), (64, 32), (128, 64)],
        activations=["relu"],
        solvers=["adam", "lbfgs"],
        alphas=[1e-4, 1e-2],
        learning_rates=["constant"],
        learning_rate_inits=(0.001,),
    ),
]).sort_values(by="val_log_loss").reset_index(drop=True))
print(results.head(10).to_string())

  hidden_layer_sizes activation solver   alpha  learning_rate_init learning_rate  n_iter  train_log_loss  val_log_loss  train_auc   val_auc
0              (64,)       relu    sgd  0.0001               0.001      constant      50        0.687490      0.690847   0.560322  0.540640
1              (64,)       relu    sgd  0.0100               0.001      constant      84        0.686642      0.690869   0.564481  0.541270
2              (64,)       relu    sgd  0.0100               0.001      adaptive     139        0.686642      0.690869   0.564481  0.541270
3              (64,)       relu    sgd  0.0001               0.001      adaptive     111        0.687169      0.690895   0.561888  0.540753
4              (64,)       relu    sgd  0.0100               0.010      constant      26        0.685820      0.691127   0.567891  0.541304
5              (64,)       relu    sgd  0.0001               0.010      constant      26        0.685847      0.691220   0.567681  0.541154
6          (128, 64)

In [23]:
best = results.iloc[0]
evaluate_model('MLP', MLPClassifier(hidden_layer_sizes=best.hidden_layer_sizes, activation=best.activation, solver=best.solver, alpha=best.alpha, learning_rate=best.learning_rate, learning_rate_init=best.learning_rate_init, max_iter=200, early_stopping=True, random_state=42), X_train_sample, y_train_sample, X_val, y_val)

{'accuracy': 0.5278421905276198,
 'confusion_matrix': [[47432, 45838], [43346, 52270]],
 'f1': 0.5396337056843757,
 'fit_time': 8.12374616600573,
 'log_loss': 0.6908474849818931,
 'model': 'MLP',
 'precision': 0.5327802014106903,
 'predict_time': 0.025135541975032538,
 'recall': 0.5466658299866132,
 'roc_auc': 0.5406395979454073,
 'train_log_loss': 0.6874901398089939}


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: Us

## Second round: smaller networks, stronger regularization, full training data

In [24]:
# Round two: all 1.1M rows (evaluate_mlp_full fits on X_train). Round one peaked at the smallest network and the
# strongest alpha tried, so this round goes smaller and stronger, keeping round one's sgd / constant 0.001
results = search("mlp_full", lambda: select_mlp_full(
    hidden_layer_sizes_list=[(8,), (16,), (32,), (64,)],
    alphas=[0.01, 0.03, 0.1, 0.3, 1.0],
))
print(results.to_string())

   hidden_layer_sizes  alpha  n_iter  train_log_loss  val_log_loss  train_auc   val_auc
0               (64,)   0.03      33        0.687029      0.690187   0.562360  0.544088
1               (16,)   0.10      54        0.687124      0.690241   0.561976  0.543324
2               (64,)   0.30      46        0.687314      0.690245   0.561922  0.542085
3               (16,)   0.03      66        0.687198      0.690328   0.561455  0.543510
4               (32,)   0.30      51        0.687450      0.690341   0.560357  0.541823
5               (32,)   0.10      83        0.685745      0.690363   0.568344  0.543779
6               (16,)   0.30      36        0.687937      0.690377   0.558026  0.541243
7               (16,)   0.01      90        0.686954      0.690492   0.562491  0.543643
8               (32,)   1.00      17        0.689115      0.690551   0.551317  0.538770
9                (8,)   0.30      43        0.687974      0.690551   0.557642  0.540385
10              (16,)   1.00    

In [25]:
best = results.iloc[0]
evaluate_model('MLP (full data)', MLPClassifier(hidden_layer_sizes=best.hidden_layer_sizes, solver="sgd", learning_rate="constant", learning_rate_init=0.001, alpha=best.alpha, max_iter=200, early_stopping=True, random_state=42), X_train, y_train, X_val, y_val)

{'accuracy': 0.5296104528657497,
 'confusion_matrix': [[49219, 44051], [44799, 50817]],
 'f1': 0.5335566241784087,
 'fit_time': 30.834168541012332,
 'log_loss': 0.6901865536197398,
 'model': 'MLP (full data)',
 'precision': 0.5356600750516507,
 'predict_time': 0.024265707994345576,
 'recall': 0.5314696285140562,
 'roc_auc': 0.5440877036073962,
 'train_log_loss': 0.6870289504920708}


# Kernel SVM (RBF)

In [26]:
# a first search (C 0.1-10, gamma 0.01-0.1) peaked at the smallest C and gamma, so both go lower
results = search("kernel_svm", lambda: select_kernel_svm(Cs=[0.01, 0.03, 0.1, 1.0], gammas=[0.001, 0.003, 0.01, 'scale']))
print(results.to_string())

       C  gamma  n_support  train_log_loss  val_log_loss  train_auc   val_auc
0   0.10   0.01      19389        0.688973      0.690666   0.553921  0.539057
1   1.00   0.01      19158        0.685245      0.690673   0.586987  0.539234
2   0.10  scale      19348        0.686469      0.690703   0.578382  0.538985
3   0.03   0.01      19613        0.689805      0.690709   0.547161  0.538460
4   0.10  0.003      19537        0.689813      0.690741   0.546808  0.538035
5   0.10  0.001      19684        0.690257      0.690793   0.543582  0.537759
6   1.00  0.003      19266        0.688579      0.690815   0.556826  0.537862
7   0.03  0.003      19728        0.690316      0.690832   0.543958  0.538377
8   0.01  0.003      19735        0.690371      0.690868   0.543711  0.538383
9   0.01   0.01      19737        0.690235      0.690931   0.545162  0.538394
10  0.01  0.001      19735        0.690510      0.690942   0.542774  0.537740
11  0.03  0.001      19735        0.690518      0.690944   0.542

In [27]:
best = results.iloc[0]
evaluate_model('Kernel SVM', SVC(kernel="rbf", C=best.C, gamma=best.gamma, probability=True, random_state=42), X_train_small, y_train_small, X_val, y_val)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


... (12 outputs left out)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


{'accuracy': 0.5258198066558665,
 'confusion_matrix': [[36705, 56565], [33001, 62615]],
 'f1': 0.5830183057412615,
 'fit_time': 29.87447445804719,
 'log_loss': 0.6911546909934732,
 'model': 'Kernel SVM',
 'precision': 0.5253817754656822,
 'predict_time': 104.53250254195882,
 'recall': 0.6548590194109772,
 'roc_auc': 0.5353807988422364,
 'train_log_loss': 0.6889728793537646}


# Bayesian Network

In [28]:
# Round one: 200k-row sample (evaluate_bn fits on X_train_sample), every structure method and discretization
results = search("bayesian_network", lambda: select_bn(
    learning_methods=["NaiveBayes", "TAN", "Chow-Liu", "MIIC", "GHC"],
    discretizations=["quantile", "uniform"],
    n_bins=[3, 5, 10],
))
print(results.to_string())

   learning_method discretization  n_bins  train_log_loss  val_log_loss  train_auc   val_auc
0              TAN        uniform       3        0.690587      0.691612   0.540815  0.532482
1              TAN        uniform       5        0.689053      0.691742   0.552591  0.539492
2              GHC        uniform       5        0.691505      0.691893   0.531487  0.526736
3              GHC       quantile       3        0.689974      0.691960   0.544369  0.529277
4             MIIC        uniform       5        0.691663      0.692041   0.528251  0.523086
5              GHC       quantile       5        0.689783      0.692106   0.546205  0.528367
6         Chow-Liu       quantile      10        0.690886      0.692108   0.537925  0.528217
7              GHC       quantile      10        0.690886      0.692108   0.537925  0.528217
8         Chow-Liu        uniform       3        0.692204      0.692419   0.523884  0.519401
9              TAN       quantile       3        0.689280      0.69252

In [29]:
best = results.iloc[0]
evaluate_model('Bayesian Network', make_bn(best.learning_method, best.discretization, int(best.n_bins)), X_train_sample, y_train_sample, X_val, y_val)

{'accuracy': 0.5210973814893639,
 'confusion_matrix': [[50110, 43160], [47298, 48318]],
 'f1': 0.5165104172234278,
 'fit_time': 39.99881991697475,
 'log_loss': 0.6924038303947364,
 'model': 'Bayesian Network',
 'precision': 0.5281925708913618,
 'predict_time': 28.68626425002003,
 'recall': 0.5053338353413654,
 'roc_auc': 0.529978101612967,
 'train_log_loss': 0.6905873325979008}


## Second round: the best structure methods on the full training data

In [30]:
# Round two: all 1.1M rows (evaluate_bn_full fits on X_train). Round one ranked TAN and GHC first and equal-width
# (uniform) bins above equal-frequency for both, with the best at 3 bins, the fewest tried, so 2 bins is added
results = search("bayesian_network_full", lambda: select_bn_full(
    learning_methods=["TAN", "GHC"],
    discretizations=["uniform"],
    n_bins=[2, 3, 5],
))
print(results.to_string())

  learning_method discretization  n_bins  train_log_loss  val_log_loss  train_auc   val_auc
0             GHC        uniform       5        0.690923      0.691519   0.536804  0.532233
1             GHC        uniform       2        0.689371      0.691967   0.549082  0.532335
2             TAN        uniform       3        0.690835      0.691979   0.539084  0.531885
3             GHC        uniform       3        0.691509      0.692220   0.529678  0.523674
4             TAN        uniform       5        0.690331      0.692375   0.547658  0.538230
5             TAN        uniform       2        0.689806      0.692521   0.548590  0.534713


In [31]:
best = results.iloc[0]
evaluate_model('Bayesian Network (full data)', make_bn(best.learning_method, best.discretization, int(best.n_bins)), X_train, y_train, X_val, y_val)

{'accuracy': 0.5228921148205796,
 'confusion_matrix': [[45969, 47301], [42818, 52798]],
 'f1': 0.5395396367166543,
 'fit_time': 56.526676666981075,
 'log_loss': 0.6915187573538829,
 'model': 'Bayesian Network (full data)',
 'precision': 0.5274578167614062,
 'predict_time': 1.4414508750196546,
 'recall': 0.5521879183400268,
 'roc_auc': 0.5322331797975649,
 'train_log_loss': 0.6906390499193449}


# Comparison

Every model refitted with its best grid setting and scored on the full validation set. KNN, MLP and the Bayesian network
were trained on the 200k sample, the kernel SVM on 20k rows.

In [32]:
comparison = pd.DataFrame(model_results).sort_values("log_loss").reset_index(drop=True)
save(comparison, "comparison")
print(comparison.drop(columns="confusion_matrix").to_string())

                           model     fit_time  predict_time  accuracy  precision    recall        f1   roc_auc  train_log_loss  log_loss
0                MLP (full data)    30.834169      0.024266  0.529610   0.535660  0.531470  0.533557  0.544088        0.687029  0.690187
1                  Random Forest    19.251790      0.173115  0.527779   0.532565  0.549029  0.540672  0.538751        0.684907  0.690679
2                            KNN     0.011335     55.821442  0.526254   0.529977  0.566903  0.547819  0.537682        0.000010  0.690744
3            Logistic Regression     0.331401      0.008365  0.527360   0.529811  0.589305  0.557977  0.537722        0.689594  0.690824
4              Linear Regression     0.313166      0.008045  0.527053   0.529734  0.585352  0.556156  0.537809        0.689584  0.690840
5                            MLP     8.123746      0.025136  0.527842   0.532780  0.546666  0.539634  0.540640        0.687490  0.690847
6                     Linear SVM   445.11

# Variability of the top 3

The top models are within a fraction of a percent of each other, so: how stable is each score, and is the ranking real?

In [33]:
top3 = [m for m in comparison["model"] if m != "Majority"][:3]
y = y_val.to_numpy()

# 1. Bootstrap the validation set: spread of each score, and how often each model comes out best
rng = np.random.RandomState(0)
boot = []
for _ in range(200):
    i = rng.randint(0, len(y), len(y))
    boot.append({name: log_loss(y[i], val_scores[name][i]) for name in top3})
boot = pd.DataFrame(boot)
bootstrap = pd.DataFrame({
    "log_loss": [log_loss(y, val_scores[n]) for n in top3],
    "ci_low": boot.quantile(0.025),
    "ci_high": boot.quantile(0.975),
    "times_best": boot.idxmin(axis=1).value_counts().reindex(top3, fill_value=0) / len(boot),
})
save(bootstrap.rename_axis("model").reset_index(), "top3_bootstrap")
print(bootstrap.to_string())

                 log_loss    ci_low   ci_high  times_best
MLP (full data)  0.690187  0.689748  0.690602         1.0
Random Forest    0.690679  0.690334  0.691011         0.0
KNN              0.690744  0.690438  0.690990         0.0


In [34]:
# 2. Retrain with different random seeds (models with no randomness show 0 spread)
# epochs = training passes before early stopping (MLP only); the "search" row is the fitted model from the search (seed 42)
rows = []
for name in top3:
    model, Xtr, ytr = fits[name]
    rows.append({"model": name, "seed": "search", "log_loss": log_loss(y, val_scores[name]), "auc": roc_auc_score(y, val_scores[name]),
                 "epochs": getattr(model, "n_iter_", None)})
    seed_keys = [k for k in model.get_params() if k.endswith("random_state")]
    for seed in range(5):
        m = clone(model).set_params(**{k: seed for k in seed_keys}).fit(Xtr, ytr)
        p = np.clip(get_scores(m, X_val), 1e-15, 1 - 1e-15)
        rows.append({"model": name, "seed": seed, "log_loss": log_loss(y, p), "auc": roc_auc_score(y, p), "epochs": getattr(m, "n_iter_", None)})
seeds = pd.DataFrame(rows)
save(seeds, "top3_seeds")
print(seeds[seeds.seed != "search"].groupby("model")[["log_loss", "auc"]].agg(["mean", "std"]).to_string(), "\n")
print(seeds.to_string())

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


... (41 outputs left out)

/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


/Users/yomna/Documents/projects/crypto-predictor/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


                 log_loss                 auc          
                     mean       std      mean       std
model                                                  
KNN              0.690744  0.000000  0.537682  0.000000
MLP (full data)  0.691982  0.000306  0.540376  0.000597
Random Forest    0.690693  0.000012  0.538700  0.000132 

              model    seed  log_loss       auc  epochs
0   MLP (full data)  search  0.690187  0.544088    33.0
1   MLP (full data)       0  0.691611  0.540791    94.0
2   MLP (full data)       1  0.692242  0.540247   129.0
3   MLP (full data)       2  0.692101  0.539958   140.0
4   MLP (full data)       3  0.691702  0.541168    93.0
5   MLP (full data)       4  0.692257  0.539717   138.0
6     Random Forest  search  0.690679  0.538751     NaN
7     Random Forest       0  0.690679  0.538775     NaN
8     Random Forest       1  0.690684  0.538844     NaN
9     Random Forest       2  0.690697  0.538492     NaN
10    Random Forest       3  0.690706  0.53869

In [35]:
# 3. Per coin and 4. per month of the validation period
symbol = X_val.filter(like="symbol=").idxmax(axis=1).str.removeprefix("symbol=").to_numpy()
month = meta_val["open_time"].dt.strftime("%Y-%m").to_numpy()

def breakdown(groups):
    rows = []
    for g in np.unique(groups):
        mask = groups == g
        rows.append({"group": g, "rows": mask.sum(), "up_share": y[mask].mean(),
                     **{f"{n} auc": roc_auc_score(y[mask], val_scores[n][mask]) for n in top3}})
    return pd.DataFrame(rows)

per_symbol, per_month = breakdown(symbol), breakdown(month)
save(per_symbol, "top3_per_symbol")
save(per_month, "top3_per_month")
print(per_symbol.to_string(), "\n")
print(per_month.to_string())

      group   rows  up_share  MLP (full data) auc  Random Forest auc   KNN auc
0   ADAUSDT  20545  0.497542             0.545371           0.539849  0.538435
1  AVAXUSDT  20930  0.498328             0.538273           0.531722  0.530017
2   BNBUSDT  15228  0.516942             0.539439           0.536344  0.531644
3   BTCUSDT  14135  0.518359             0.545424           0.541069  0.540201
4  DOGEUSDT  20627  0.507151             0.542214           0.540211  0.539178
5   ETHUSDT  17586  0.517286             0.557067           0.548214  0.548163
6  LINKUSDT  20876  0.496455             0.536919           0.535377  0.534418
7   LTCUSDT  19704  0.505380             0.547625           0.544529  0.546218
8   SOLUSDT  20011  0.507271             0.539873           0.531778  0.532553
9   XRPUSDT  19244  0.505820             0.545749           0.539828  0.537299 

     group   rows  up_share  MLP (full data) auc  Random Forest auc   KNN auc
0  2024-10  18999  0.512237             0.545405   

In [36]:
# 5. Train vs validation: a big gap means the model memorised noise from the training period
print(comparison[comparison["model"].isin(top3)][["model", "train_log_loss", "log_loss", "fit_time", "predict_time"]].to_string())

             model  train_log_loss  log_loss   fit_time  predict_time
0  MLP (full data)        0.687029  0.690187  30.834169      0.024266
1    Random Forest        0.684907  0.690679  19.251790      0.173115
2              KNN        0.000010  0.690744   0.011335     55.821442


# Final model on the test set

Random Forest. The top three on validation are the MLP (full data), Random Forest and KNN, but the MLP's lead comes from
a lucky seed: with seed 42 early stopping happened to stop after 33 epochs, while seeds 0-4 train longer and average a
validation log loss of 0.6920, worse than Random Forest (0.6907, nearly identical across seeds). Early stopping monitors a
random 10% of the training period, so it keeps training on patterns that do not carry over to later months.

Order of events, for the record: Random Forest was chosen first and tested once; the MLP second round then beat it on
validation by more than the bootstrap noise and was tested once (out/results/test_mlp.csv: accuracy 0.521, AUC 0.529);
the seed analysis above then showed that the bootstrap rule had ignored training randomness, so we returned to Random
Forest. Its test score below is the same deterministic number as its first evaluation.

In [37]:
final_name = "Random Forest"
model, _, _ = fits[final_name]
p_test = np.clip(get_scores(model, X_test), 1e-15, 1 - 1e-15)
test_row = {
    "model": final_name,
    "rows": len(y_test),
    "up_share": y_test.mean(),
    "accuracy": accuracy_score(y_test, (p_test > 0.5).astype(int)),
    "roc_auc": roc_auc_score(y_test, p_test),
    "log_loss": log_loss(y_test, p_test),
    "prior_log_loss": log_loss(y_test, np.full(len(y_test), y_train.mean())),  # the Majority baseline on test
    "confusion_matrix": confusion_matrix(y_test, (p_test > 0.5).astype(int)).tolist(),  # rows: true down/up, cols: predicted
    "test_start": meta_test["open_time"].min(),
    "test_end": meta_test["open_time"].max(),
}
save(pd.DataFrame([test_row]), "test")
pprint.pprint(test_row)

{'accuracy': 0.5239982740108176,
 'confusion_matrix': [[74160, 75493], [66812, 82494]],
 'log_loss': 0.6920019986429594,
 'model': 'Random Forest',
 'prior_log_loss': 0.6932587223900252,
 'roc_auc': 0.5320347642618501,
 'rows': 298959,
 'test_end': Timestamp('2026-10-31 22:45:00'),
 'test_start': Timestamp('2025-07-01 00:00:00'),
 'up_share': np.float64(0.49941965286209816)}


## What the final model uses

Permutation importance on the validation set: how much the log loss rises when one feature is shuffled, which breaks its
link with the label. The 10 one-hot coin columns are shuffled together as one group ("coin"), since together they encode
a single input: which pair the row belongs to.

In [38]:
groups = {c: [c] for c in X_val.columns if not c.startswith("symbol=")}
groups["coin (10 one-hot columns)"] = [c for c in X_val.columns if c.startswith("symbol=")]
base = log_loss(y_val, val_scores[final_name])
rng = np.random.RandomState(0)
rows = []
for name, cols in groups.items():
    rises = []
    for _ in range(3):
        X_perm = X_val.copy()
        X_perm[cols] = X_val[cols].to_numpy()[rng.permutation(len(X_val))]  # shuffle whole rows of the group together
        rises.append(log_loss(y_val, np.clip(get_scores(model, X_perm), 1e-15, 1 - 1e-15)) - base)
    rows.append({"feature": name, "importance": np.mean(rises), "std": np.std(rises)})
importance = to_df(rows, by="importance", ascending=False)
save(importance, "feature_importance")
print(importance.to_string())

                      feature    importance           std
0                      ret_96  4.309084e-04  4.463314e-05
1                   btc_ret_4  2.916000e-04  5.865393e-05
2                       ret_4  2.418122e-04  4.263473e-05
3                  btc_ret_16  2.297331e-04  3.178271e-05
4                    hour_cos  1.967121e-04  1.398799e-05
5                  body_ratio  1.507164e-04  4.098656e-06
6                      ret_16  1.481131e-04  1.911390e-05
7                     dow_sin  1.227980e-04  7.983487e-06
8                     returns  1.105175e-04  2.066867e-05
9                     dow_cos  9.412424e-05  1.089127e-05
10                    ret_672  6.742262e-05  7.871722e-06
11                   hour_sin  3.282178e-05  1.120143e-05
12         taker_buy_ratio_16  2.243326e-05  5.499194e-06
13  coin (10 one-hot columns)  1.835335e-05  3.413783e-06
14            taker_buy_ratio  1.733345e-05  6.052029e-06
15            relative_volume  9.268640e-06  6.057036e-06
16            